# Earth Trend Detective: ETG Trend Microscope (`etg-trend-microscope-v1`)

This notebook evaluates and exports `etg-trend-microscope-v1`, a multi-task model built to catch early climate trend breaks and filter out spurious correlations on Earth Trend Graphs:

1. **Early Onset (`inflection_head`):** Flags emerging slope breaks 3 to 6 steps before rolling Mann-Kendall crosses `|Z| >= 1.96`, while ignoring AR(1) red noise.
2. **Directional Coupling (`coupling_head`):** Separates real lead-lag (`tau = 0..5`) and opposite-trend couplings from seasonal hemisphere illusions and shared drift.
3. **Cycle De-Aliasing & Trust (`dealias_head`, `trust_gate`):** Splits each time series into `[secularShare, oscillationShare, artifactShare]` with a `falseTrendProb` score and downweights noisy sensor glitches.

On CUDA/Blackwell runtimes it trains a Transformer teacher (`d_model=128`, `bf16` + `torch.compile`) and distills portable JSON weights (`etg-trend-microscope-v1.json`); on CPU runtimes it loads the tracked distilled weights and runs the full holdout and multi-agency benchmarks.

In [1]:
import base64
import importlib
import json
import sys
import zlib
from pathlib import Path

ML_DIR = Path("/content/ml") if Path("/content").exists() else Path(".data/colab_ml")
ML_DIR.mkdir(parents=True, exist_ok=True)
(ML_DIR / "artifacts").mkdir(parents=True, exist_ok=True)

PAYLOAD_B64 = """
...
"""

files = json.loads(zlib.decompress(base64.b64decode("".join(PAYLOAD_B64.split()))).decode("utf-8"))
for rel_name, source in files.items():
    target = ML_DIR / rel_name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")

if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

for mod_name in ("etg_trend_microscope", "harvest_global_multiscale"):
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

print(f"Unpacked {len(files)} files ({list(files.keys())[:4]} + ...) -> {ML_DIR}")
try:
    import torch
    print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        cap = torch.cuda.get_device_capability(0)
        bf16_ok = torch.cuda.is_bf16_supported()
        torch.set_float32_matmul_precision("high")
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU: {props.name} | SM {cap[0]}.{cap[1]} | VRAM {vram_gb:.1f} GB | bf16={bf16_ok}")
    else:
        print("CPU mode detected; loading canonical Blackwell bf16 distilled weights for live evaluation.")
except Exception as exc:
    print(f"PyTorch unavailable ({exc}); using pure-Python PortableTrendMicroscope.")


Unpacked 12 files (['etg_trend_microscope.py', 'harvest_global_multiscale.py', 'microscope/__init__.py', 'microscope/corpus_and_benchmarks.py'] + ...) -> .data/colab_ml


PyTorch: 2.15.0 | CUDA available: False
CPU mode detected; loading canonical Blackwell bf16 distilled weights for live evaluation.


In [2]:
import os
import sys
import time
from pathlib import Path

ML_DIR = Path("/content/ml") if Path("/content").exists() else Path(".data/colab_ml")
ARTIFACTS_DIR = ML_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

from harvest_global_multiscale import (
    build_synthetic_global_multiscale_bundle,
    harvest_live_global_multiscale,
)
from etg_trend_microscope import build_microscope_training_corpus

t0 = time.perf_counter()
live_path = ARTIFACTS_DIR / "etg-ml-v2-live.json"
try:
    if os.environ.get("ETG_OFFLINE") == "1":
        raise RuntimeError("Offline deterministic mode (ETG_OFFLINE=1)")
    live_raw = harvest_live_global_multiscale(
        output_path=str(live_path),
        start_year=2000,
        end_year=2025,
        horizon_steps=1,
    )
    if not live_raw.get("node_ids"):
        raise RuntimeError("Empty node_ids from live harvester")
except Exception as exc:
    print(f"Live harvest fallback triggered ({exc}); building synthetic benchmark bundle.")
    live_raw = build_synthetic_global_multiscale_bundle(
        num_months=84,
        horizon_steps=1,
        seed=42,
    )

corpus = build_microscope_training_corpus(
    live_bundle=live_raw,
    num_synthetic_graphs=320,
    seed=42,
)

meta = live_raw.get("harvester_metadata", {})
print(f"Bundle Mode        : {meta.get('mode', 'synthetic')}")
print(f"Live Sources       : {meta.get('live_sources', [])}")
print(f"Bundle Nodes/Steps : {len(live_raw.get('node_ids', []))} nodes / {len(live_raw.get('time_index', []))} steps")
print(f"Microscope Corpus  : train={len(corpus['train'])} graphs | val={len(corpus['val'])} graphs")
print(f"Corpus preparation finished in {time.perf_counter() - t0:.2f}s")

Bundle Mode        : live
Live Sources       : ['noaa_co2', 'nasa_power', 'nasa_eonet']
Bundle Nodes/Steps : 8 nodes / 312 steps
Microscope Corpus  : train=264 graphs | val=88 graphs
Corpus preparation finished in 7.38s


In [3]:
import json
import time
from pathlib import Path
import etg_trend_microscope as tm
from etg_trend_microscope import PortableTrendMicroscope, evaluate_microscope_benchmarks

t0 = time.perf_counter()
train_res = None
if getattr(tm, "TORCH_AVAILABLE", False) and tm.torch.cuda.is_available():
    train_res = tm.train_blackwell_microscope_and_distill(
        corpus,
        teacher_epochs=50,
        distill_epochs=65,
        d_model=128,
        lr=1.8e-3,
        compile_model=True,
    )
    teacher_model = train_res["teacher"]
    student_model = train_res["student"]
    portable_model = train_res["portable_model"]
    portable_weights = train_res["portable_weights"]
    benchmarks = train_res["benchmarks"]
    print(f"Backend       : PyTorch Teacher + Distilled Portable ({train_res['device']}, bf16={train_res['bf16']})")
    print(f"Teacher Loss  : start={train_res['teacher_loss_history'][0]:.4f} -> end={train_res['teacher_loss_history'][-1]:.4f}")
    print(f"Distill Loss  : start={train_res['distill_loss_history'][0]:.4f} -> end={train_res['distill_loss_history'][-1]:.4f}")
else:
    teacher_model = None
    student_model = None
    weights_path = None
    for cand in (
        Path("ml/weights/etg-trend-microscope-v1.json"),
        Path("../ml/weights/etg-trend-microscope-v1.json"),
        ML_DIR / "weights/etg-trend-microscope-v1.json",
    ):
        if cand.exists():
            weights_path = cand
            break

    if weights_path is not None:
        portable_weights = json.loads(weights_path.read_text(encoding="utf-8"))
        portable_model = PortableTrendMicroscope.from_weights_dict(portable_weights)
        tsum = portable_weights.get("training_summary", {})
        benchmarks = tsum.get("benchmarks") or evaluate_microscope_benchmarks(portable_model, corpus["val"])
        val_check = evaluate_microscope_benchmarks(portable_model, corpus["val"])
        print(
            f"Backend       : Loaded Distilled Portable Weights ({weights_path}, "
            f"distilled={tsum.get('device', 'cuda')}, bf16={tsum.get('bf16', True)})"
        )
        print(
            f"Corpus Forward Check (val n={val_check['val_graphs']}): "
            f"onset_auroc={val_check['early_onset_auroc']:.4f} | "
            f"coupling_auroc={val_check['true_link_auroc']:.4f} | "
            f"illusion_rej={val_check['illusion_rejection_rate']:.4f}"
        )
    else:
        portable_model = PortableTrendMicroscope(hidden_dim=32, seed=42)
        fit_info = portable_model.fit_portable(corpus["train"], epochs=42, lr=0.045, seed=42, batch_size=6)
        benchmarks = evaluate_microscope_benchmarks(portable_model, corpus["val"])
        portable_model.training_summary = {"backend": "portable_adam_cpu_fallback", **fit_info, "benchmarks": benchmarks}
        portable_weights = portable_model.to_weights_dict()
        print("Backend       : Pure-Python PortableTrendMicroscope (CPU fallback)")
        print(f"Loss Curve    : start={fit_info['loss_history'][0]:.4f} -> end={fit_info['loss_history'][-1]:.4f}")

print(f"Stage 3 completed in {time.perf_counter() - t0:.2f}s")


Backend       : Loaded Distilled Portable Weights (../ml/weights/etg-trend-microscope-v1.json, distilled=cuda, bf16=True)
Corpus Forward Check (val n=88): onset_auroc=0.9789 | coupling_auroc=0.9916 | illusion_rej=0.9894
Stage 3 completed in 0.37s


In [4]:
import json
from pathlib import Path
import etg_trend_microscope as tm

print("=== ETG Trend Microscope (`etg-trend-microscope-v1`) Holdout Benchmarks ===")
print(
    f"1. Early Onset AUROC     : {benchmarks['early_onset_auroc']:.4f} "
    f"(vs Classical Mann-Kendall AUROC: {benchmarks['classical_mk_auroc']:.4f})"
)
print(
    f"2. Pre-MK Fresh Break Rec: {benchmarks['pre_mk_recall_on_fresh_breaks']:.4f} "
    f"(vs Classical Mann-Kendall Recall: {benchmarks['classical_mk_recall_on_fresh_breaks']:.4f})"
)
print(
    f"3. True Link AUROC       : {benchmarks['true_link_auroc']:.4f} "
    f"(vs Raw Pearson |r| AUROC      : {benchmarks['raw_pearson_auroc']:.4f})"
)
print(f"4. Illusion Rejection    : {benchmarks['illusion_rejection_rate']:.4f}")
print(f"5. De-Alias Simplex MAE  : {benchmarks['dealias_simplex_mae']:.4f}")
print(f"6. False-Trend Detect F1 : {benchmarks['false_trend_f1']:.4f}")

hist = tm.evaluate_real_historical_microscope(portable_model)
print(
    f"\n=== Real Historical Multi-Agency Benchmark (NASA GISTEMP v4 / NOAA CO2 / NSIDC / KNMI) | "
    f"Checks Passed: {hist['secularChecksPassed']}/{hist['secularChecksTotal']} ==="
)
for uid, info in hist["nodes"].items():
    deal = info.get("dealiasing", {})
    print(
        f"  {info.get('name', uid):<22} | dir={info.get('emergingDirection', '?'):<10} | "
        f"secularShare={deal.get('secularShare', 0):.3f} | falseTrendProb={deal.get('falseTrendProb', 0):.3f}"
    )

live_scan = portable_model.scan_v2_bundle(live_raw, top_k=8)
print("\n=== Live Bundle AI Perception Brief ===")
print(live_scan["aiPerceptionBrief"])

ML_DIR = Path("/content/ml") if Path("/content").exists() else Path(".data/colab_ml")
ARTIFACTS_DIR = ML_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
json_path = ARTIFACTS_DIR / "etg-trend-microscope-v1.json"
json_path.write_text(json.dumps(portable_weights, indent=2), encoding="utf-8")

pt_path = ARTIFACTS_DIR / "etg_trend_microscope_v1.pt"
onnx_path = ARTIFACTS_DIR / "etg_trend_microscope_v1.onnx"
if teacher_model is not None and getattr(tm, "TORCH_AVAILABLE", False):
    import torch
    torch.save(
        {
            "schema_version": "etg-trend-microscope-v1",
            "teacher_state_dict": teacher_model.state_dict(),
            "student_state_dict": student_model.state_dict() if student_model is not None else None,
            "benchmarks": benchmarks,
        },
        pt_path,
    )
    try:
        tm.export_microscope_onnx(teacher_model, corpus["val"][0], onnx_path)
        print(f"Exported ONNX Teacher  -> {onnx_path}")
    except Exception as exc:
        print(f"ONNX export skipped ({exc})")
    print(f"Exported PyTorch Ckpt  -> {pt_path}")
print(f"Exported JSON Weights  -> {json_path}")
print(
    f"Verified portable schema={portable_weights['schema_version']} | "
    f"size={json_path.stat().st_size:,} bytes | hidden_dim={portable_weights['hidden_dim']}"
)


=== ETG Trend Microscope (`etg-trend-microscope-v1`) Holdout Benchmarks ===
1. Early Onset AUROC     : 0.9789 (vs Classical Mann-Kendall AUROC: 0.4442)
2. Pre-MK Fresh Break Rec: 0.7959 (vs Classical Mann-Kendall Recall: 0.0000)
3. True Link AUROC       : 0.9916 (vs Raw Pearson |r| AUROC      : 0.7010)
4. Illusion Rejection    : 0.9894
5. De-Alias Simplex MAE  : 0.0570
6. False-Trend Detect F1 : 0.9380

=== Real Historical Multi-Agency Benchmark (NASA GISTEMP v4 / NOAA CO2 / NSIDC / KNMI) | Checks Passed: 6/6 ===
  NOAA Global CO2        | dir=increasing | secularShare=0.789 | falseTrendProb=0.000
  NASA GISTEMP Global    | dir=increasing | secularShare=0.785 | falseTrendProb=0.000
  NASA GISTEMP 24N-44N   | dir=increasing | secularShare=0.756 | falseTrendProb=0.000
  NASA GISTEMP 44N-64N   | dir=increasing | secularShare=0.581 | falseTrendProb=0.003
  NSIDC Arctic Sea Ice   | dir=stable     | secularShare=0.328 | falseTrendProb=0.146
  KNMI NAO Mode          | dir=stable     | secular